# 03｜开发首个因子并接入策略
因子示例：`ma_gap_20 = close / 最近20期收盘均价 - 1`。这是教学例子，不是投资建议。本页先手算并核对时序，随后运行原有规则与教学信号两个短区间实验。

In [1]:
from pathlib import Path
import os, sys, subprocess, json, pandas as pd
ROOT = next((p.resolve() for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "qlib_quant").is_dir()), None)
if ROOT is None: raise RuntimeError("请打开 D:/qlib_quant 项目目录。")
sys.path.insert(0, str(ROOT / "src"))
from qlib_quant.config import effective_config
from qlib_quant.data.db import list_codes, iter_quotes
from qlib_quant.factors.technical import add_technical_features
from qlib_quant.signals.legacy_v1 import score_signals, legacy_signal_names
from qlib_quant.experiments import read_json, read_csv
cfg = effective_config()
DB = Path(cfg["data"]["database"])
assert DB.exists(), f"找不到练习数据库：{DB}"
print("项目:", ROOT, "数据库:", DB)

项目: D:\qlib_quant 数据库: D:\quanttrade20260207\data\stockdata\daily_hfq_repaired.db


## 1. 手工算一个例子
最近 20 期均价为 12，当前收盘价为 12.6，偏离率为 5%。

In [2]:
close, average_20 = 12.6, 12.0
manual_gap = close / average_20 - 1
print(f"均价={average_20:.2f} 偏离率={manual_gap:.4f} ({manual_gap:.2%})")
assert abs(manual_gap - 0.05) < 1e-12

均价=12.00 偏离率=0.0500 (5.00%)


In [3]:
def calculate_ma_gap(close_series, window=20):
    """计算收盘价相对 rolling mean 的偏离；不足 window 期时返回 NaN。"""
    moving_average = close_series.rolling(window=window, min_periods=window).mean()
    return close_series / moving_average - 1

example_prices = pd.Series(range(1, 21), dtype=float)
example_gap = calculate_ma_gap(example_prices, window=20)
print("第19项:", example_gap.iloc[18], "第20项:", example_gap.iloc[19])
assert example_gap.iloc[:19].isna().all()
assert example_gap.iloc[19] == example_prices.iloc[19] / example_prices.mean() - 1

第19项: nan 第20项: 0.9047619047619047


## 2. 用行情检查代码里的因子
第 20 个观察之前没有完整均线，偏离率应为空。因子按每只股票分组计算。

In [4]:
codes = list_codes(DB, "2020-01-01", "2020-02-28", max_stocks=5)
raw = pd.DataFrame(list(iter_quotes(DB, "2019-01-01", "2020-02-28", codes=codes)))
features = add_technical_features(raw, windows=(20,))
first_code = str(codes[0])
one = features[features.code == first_code].sort_values("date")
display(one[["code","date","close","ma_20","ma_gap_20"]].tail(8))
assert one.ma_gap_20.iloc[:19].isna().all()

,code,date,close,ma_20,ma_gap_20
272,000001,2020-02-19,2395.47,2400.5890,-0.002132
273,000001,2020-02-20,2448.16,2393.5890,0.022799
274,000001,2020-02-21,2446.65,2387.9440,0.024584
275,000001,2020-02-24,2393.96,2379.2125,0.006198
276,000001,2020-02-25,2365.36,2368.5995,-0.001368
277,000001,2020-02-26,2357.84,2360.9975,-0.001337
278,000001,2020-02-27,2375.90,2353.6210,0.009466
279,000001,2020-02-28,2284.07,2345.7930,-0.026312


## 3. 确认没有偷看未来
把截止日以后的收盘价乘以 3，再计算因子。截止日前的输出必须完全不变。

In [5]:
cutoff = "2020-01-31"
changed = raw.copy()
changed.loc[changed.date > cutoff, "close"] *= 3
recomputed = add_technical_features(changed, windows=(20,))
left = features[features.date <= cutoff].sort_values(["code","date"]).reset_index(drop=True)
right = recomputed[recomputed.date <= cutoff].sort_values(["code","date"]).reset_index(drop=True)
pd.testing.assert_frame_equal(left, right)
print("通过：未来数据变化未影响过去因子。")

通过：未来数据变化未影响过去因子。


## 4. 从因子到信号
项目现在有一个默认关闭的教学信号 `learning_ma_gap`。规则为 `ma_gap_20 > learning_ma_gap_min`，命中加 1 分；后续排序、组合和次日开盘执行与原策略共用。

查看这些入口：
- 因子：`src/qlib_quant/factors/technical.py`
- 信号说明与预热：`src/qlib_quant/signals/legacy_v1.py`
- 默认开关与阈值：`configs/signals.yaml`
- 网页阈值：新建回测页面的“教学因子阈值”
- CLI 练习配置：`configs/learning/teaching_signal.yaml`

扩展练习：把阈值 0.02 改为 0.05，比较命中变化。改窗口前，须同步改因子、信号预热长度、说明和测试。

In [6]:
features_all = add_technical_features(raw, windows=cfg["factors"]["windows"])
old_rules = score_signals(features_all, cfg["signals"]["thresholds"], list(legacy_signal_names))
teaching = score_signals(features_all, {"learning_ma_gap_min": 0.02}, ["learning_ma_gap"])
print("原信号数:", len(legacy_signal_names),
      "教学信号命中数:", int(teaching.learning_ma_gap.sum()))
assert not score_signals(features_all).learning_ma_gap.any(), "教学信号应默认关闭"

原信号数: 11 教学信号命中数: 485


## 5. 运行两组真实小样本
下面各运行一次 5 只股票、2020-01-01 至 2020-02-28 的回测。会在 `runs/learning` 创建新结果，不覆盖旧实验。执行此单元才会启动真实后台计算。

In [ ]:
def run_experiment(config_file, output_name):
    env = os.environ.copy()
    env["PYTHONPATH"] = str(ROOT / "src")
    env["PYTHONIOENCODING"] = "utf-8"
    completed = subprocess.run(
        [sys.executable, "-m", "qlib_quant.cli", "backtest",
         "--config", str(ROOT / "configs" / "learning" / config_file),
         "--output", str(ROOT / "runs" / "learning" / output_name)],
        cwd=ROOT, env=env, capture_output=True, text=True, timeout=300
    )
    print(completed.stdout)
    if completed.returncode:
        print(completed.stderr[-6000:])
        raise RuntimeError("回测失败，请先检查上方错误和实验日志。")
    return Path(json.loads(completed.stdout.strip().splitlines()[-1])["output"])

run_a = run_experiment("baseline.yaml", "cli_baseline")
run_b = run_experiment("teaching_signal.yaml", "cli_teaching")
print("A:", run_a, "\nB:", run_b)

In [ ]:
summary = pd.DataFrame([
    {"实验":"A 原有规则", **read_json(run_a / "summary.json")},
    {"实验":"B 教学信号", **read_json(run_b / "summary.json")},
])
display(summary[["实验","start","end","cumulative_return","max_drawdown","trade_count","total_fees"]])
print("A 有效启用项:", read_json(run_a / "run_metadata.json")["effective_config"]["signals"]["enabled"])
print("B 有效启用项:", read_json(run_b / "run_metadata.json")["effective_config"]["signals"]["enabled"])
display(read_csv(run_a / "trades.csv").head())
display(read_csv(run_b / "trades.csv").head())

## 6. 独立练习与交付
1. 将教学配置阈值改成 0.05，使用新输出名再运行。
2. 在“结果与实验”比较有效参数、成交和权益。短样本收益不代表策略好坏。
3. 解释一个交易日信号的真假，或记录为何没有触发、没有成交。
4. 保存两次实验编号、唯一修改、预期和实际观察。
5. 解释完整数据流：因子 → 信号 → 排名 → 组合 → 回测。

B 没有成交时先核对参数快照、因子值与信号，不要只以交易笔数判断接入是否成功。实验通过标准是链路和账本正确，不是收益为正。

## 7. 为实验保留代码快照

结果元数据保存代码摘要；下面另将影响策略的源码与配置复制到 `runs/learning/source_snapshots`，使两次实验都能追查当时的信号和因子实现。

In [ ]:
import hashlib, shutil
snapshot_paths = [
    Path("src/qlib_quant/config.py"),
    Path("src/qlib_quant/runner.py"),
    Path("src/qlib_quant/factors/technical.py"),
    Path("src/qlib_quant/signals/legacy_v1.py"),
    Path("app/main.py"),
    Path("configs/signals.yaml"),
    Path("configs/learning/baseline.yaml"),
    Path("configs/learning/teaching_signal.yaml"),
]
digest = hashlib.sha256()
for relative in snapshot_paths:
    digest.update(relative.as_posix().encode())
    digest.update((ROOT / relative).read_bytes())
snapshot_dir = ROOT / "runs" / "learning" / "source_snapshots" / digest.hexdigest()[:12]
for relative in snapshot_paths:
    destination = snapshot_dir / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ROOT / relative, destination)
print("源码快照:", snapshot_dir)
print("快照文件数:", len(snapshot_paths))